# Verify the main CUDA benchmark numbers

`reports/TECH_REPORT.md` §4's headline speedups (1.13x–3.22x) are the main result of this submission, all measured on a Colab Tesla T4. This notebook reproduces that table directly: it clones the repo and runs `tests/run_benchmark_suite.py cuda`, which shells out to the official `torch_transformer_benchmark.py` once per config at its own full-rigor defaults (20 warmup iterations, 100 timed repeats, 3 alternating rounds) — the same invocation the grading script itself uses, not a lighter approximation.

That distinction matters here specifically: an earlier version of this same script overrode those settings to a much lighter configuration, which is the likely source of several numbers that were originally published too low (and one too high) and had to be corrected after they failed to reproduce under full rigor — see §6.3. This notebook exists so that correction, and everything after it, can be checked directly rather than taken on faith.

**Before running:** `Runtime` → `Change runtime type` → select a `T4 GPU` (or any CUDA GPU). The free tier is enough for this.

In [ ]:
# Repo path for this submission
REPO_URL = "https://github.com/brdge77e/optimized-transformer-layer.git"

!git clone "$REPO_URL" repo
%cd repo

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip install -q -r requirements.txt
import torch
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Select a GPU runtime: Runtime > Change runtime type > T4 GPU"

In [ ]:
!python3 tests/run_benchmark_suite.py cuda

## What to look for

Six configs, each printing accuracy (`PASS`/`FAIL`), baseline/optimized median latency in ms, and the resulting speedup. All six should print `PASS`, with speedups landing close to the values already published in `reports/TECH_REPORT.md` §4:

| Config | Expected CUDA speedup |
|---|---|
| small | 2.08x–2.11x |
| default | 1.13x–1.16x |
| large batch | 1.13x–1.15x |
| long seq | 3.16x–3.22x |
| causal + padding | 1.16x |
| default + `torch.compile` | 1.16x |

GPU benchmarks are noisy — Colab's shared T4s especially — so exact reproduction to three decimal places isn't expected; every one of the values above was itself re-confirmed across at least two independent runs before publishing. If a number here lands meaningfully outside its range (not just a few percent), that's worth re-running once more before assuming either this run or the published number is wrong — that's exactly the process that caught and fixed the incorrect numbers this notebook now guards against.